In [ ]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END
import operator
from langchain_core.prompts import ChatPromptTemplate
import time
import os
from langchain_openai import ChatOpenAI
from duckduckgo_search import DDGS

In [ ]:
llm = ChatOpenAI(
    model="Your_Model", 
    api_key="API_Key", 
    base_url="API_Website",
    temperature=0.7
)

In [ ]:
# ۱. تعریف ساختار داده‌ای که بین عامل‌ها جابه‌جا می‌شود
class ProposalState(TypedDict):
    requirements: str
    research_data: str
    writing_style: str
    current_draft: str
    critique: Annotated[List[str], operator.add]
    score: int
    tech_score: int
    biz_score: int
    visual_score: int
    revision_count: int

In [ ]:
# خواندن فایل اصول نگارش پروپوزال

def load_writing_guide():
    if os.path.exists("proposal_guide.txt"):
        with open("proposal_guide.txt", "r", encoding="utf-8") as f:
            return f.read()
    return "Write in a professional, academic, and formal technical tone."

# ۳. عامل جدید: (Researcher Agent)

def researcher_agent(state: ProposalState):
    print("--- 🔍 RESEARCHER AGENT IS SEARCHING WEB & DOCS ---")
    requirements = state["requirements"]
    
    try:
        with DDGS() as ddgs:
            results = [r['body'] for r in ddgs.text(f"technical implementation of {requirements}", max_results=3)]
            search_results = "\n".join(results)
    except Exception as e:
        search_results = "Could not fetch web results, relying on internal knowledge."
        
    style_guide = load_writing_guide()
    
    return {
        "research_data": search_results,
        "writing_style": style_guide
    }

# ۴. عامل نویسنده

def writer_agent(state: ProposalState):
    print("--WRITER AGENT IS GENERATING (WITH DIAGRAMS & TABLES)--")
    requirements = state["requirements"]
    research_data = state["research_data"]
    current_draft = state.get("current_draft", "")
    
    critique_history = "\n".join([f"- دور {i+1}: {c}" for i, c in enumerate(state.get("critique", []))])
    last_score = state.get("score", "None")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are an expert technical proposal writer. Your goal is to reach a score ABOVE 95/100.\n"
            "You MUST integrate the following visual elements into the proposal:\n"
            "1. A structured Markdown Table (e.g., for project phases, hardware requirements, or budget).\n"
            "2. A Mermaid.js Diagram using a ```mermaid ... ``` block representing the system architecture, "
            "data flow, or algorithm steps (e.g., Image Preprocessing -> PCA Dim Reduction -> KNN Classification).\n\n"
            "Keep the entire output highly detailed, long, academic, and structured."
        )),
        ("user", (
            "Target Subject: {requirements}\n\n"
            "Research Context: {research_data}\n\n"
            "Previous Score: {last_score}/100\n"
            "History of Critiques:\n{critique_history}\n\n"
            "Your Previous Draft:\n{current_draft}"
        ))
    ])
    
    chain = prompt | llm
    response = chain.invoke({
        "requirements": requirements, 
        "research_data": research_data,
        "last_score": last_score,
        "critique_history": critique_history, 
        "current_draft": current_draft
    })
    
    return {
        "current_draft": response.content.strip(),
        "revision_count": state.get("revision_count", 0) + 1
    }
# ۵. عامل : (Critic Agent)

def critic_agent(state: ProposalState):
    print("--CRITIC AGENT IS EVALUATING DIAGRAMS & QUALITY--")
    draft = state["current_draft"]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a strict QA manager. Critique the following technical proposal.\n"
            "CRITICAL GRADING CRITERIA:\n"
            "- It MUST contain at least one valid Markdown table (| Col 1 | Col 2 |).\n"
            "- It MUST contain a valid ```mermaid block for diagrams.\n"
            "If any of these visual elements are missing, give a score BELOW 90.\n\n"
            "⚠️ YOUR OUTPUT MUST FOLLOW THIS EXACT FORMAT:\n"
            "SCORE: [Put a single number between 0 and 100 here]\n"
            "CRITIQUE: [Your detailed feedback here]"
        )),
        ("user", "Draft to evaluate:\n\n{draft}")
    ])
    
    chain = prompt | llm
    response = chain.invoke({"draft": draft})
    output_text = response.content.strip()
    score = 80
    critique_feedback = output_text
    
    try:
        if "SCORE:" in output_text:
            score_line = [line for line in output_text.split("\n") if "SCORE:" in line][0]
            score = int(''.join(filter(str.isdigit, score_line)))
    except Exception:
        pass
        
    if "CRITIQUE:" in output_text:
        critique_feedback = output_text.split("CRITIQUE:")[-1].strip()
        
    print(f"Critic Evaluation Complete. Score: {score}/100")

    return {"critique": [critique_feedback], "score": score}

In [ ]:
# 1. منتقد فنی

def technical_critic_node(state: ProposalState):
    print("--TECHNICAL CRITIC IS EVALUATING MATH & DIMENSIONS--")
    draft = state["current_draft"]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a strict Technical Reviewer. Evaluate the proposal's engineering soundness.\n"
            "Check if it includes explicit mathematical foundations.\n"
            "If the math or architecture is missing or generic, penalize heavily.\n\n"
            "FORMAT RULE:\n"
            "SCORE: [Number between 0 and 100]\n"
            "FEEDBACK: [Technical feedback]"
        )),
        ("user", "{draft}")
    ])
    
    response = (prompt | llm).invoke({"draft": draft})
    score, feedback = parse_critic_output(response.content.strip())
    print(f"🎯 Technical Critic Score: {score}/100")
    return {"critique": [f"Technical Critic: {feedback}"], "tech_score": score}

# 2. منتقد تجاری و ساختاری

def business_critic_node(state: ProposalState):
    print("--BUSINESS CRITIC IS EVALUATING TIMELINE & VALUE--")
    draft = state["current_draft"]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a Business Consultant. Evaluate the proposal's project management and value proposition.\n"
            "Look for a clear project timeline, implementation phases, and a structured Markdown Table for requirements/budget.\n\n"
            "⚠️ FORMAT RULE:\n"
            "SCORE: [Number between 0 and 100]\n"
            "FEEDBACK: [Business feedback]"
        )),
        ("user", "{draft}")
    ])
    
    response = (prompt | llm).invoke({"draft": draft})
    score, feedback = parse_critic_output(response.content.strip())
    print(f"Business Critic Score: {score}/100")
    return {"critique": [f"Business Critic: {feedback}"], "biz_score": score}

# 3. منتقد بصری و نمودارها
def visual_critic_node(state: ProposalState):
    print("--VISUAL CRITIC IS CHECKING MERMAID DIAGRAMS--")
    draft = state["current_draft"]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a Visual Formatting Expert. Verify if the proposal contains valid visual representations.\n"
            "It MUST include a proper ```mermaid block enclosing a valid diagram (like graph TD).\n"
            "If the diagram block is missing or poorly constructed, give a very low score.\n\n"
            "⚠️ FORMAT RULE:\n"
            "SCORE: [Number between 0 and 100]\n"
            "FEEDBACK: [Visual feedback]"
        )),
        ("user", "{draft}")
    ])
    
    response = (prompt | llm).invoke({"draft": draft})
    score, feedback = parse_critic_output(response.content.strip())
    print(f"🎯 Visual Critic Score: {score}/100")
    return {"critique": [f"Visual Critic: {feedback}"], "visual_score": score}

In [ ]:
import re

def parse_critic_output(output_text: str):
    score = 75
    feedback = output_text
    
    try:
        if "SCORE:" in output_text:
            score_line = [line for line in output_text.split("\n") if "SCORE:" in line][0]
            match = re.search(r'\d+', score_line)
            if match:
                extracted_score = int(match.group())
                if 0 <= extracted_score <= 100:
                    score = extracted_score
                elif extracted_score > 100 and str(extracted_score).endswith("100"):
                    score = int(str(extracted_score)[:-3])
    except Exception as e:
        print(f"Warning during parsing score: {e}")
        
    # ۲. استخراج بخش فیدبک
    if "FEEDBACK:" in output_text:
        feedback = output_text.split("FEEDBACK:")[-1].strip()
    elif "CRITIQUE:" in output_text:
        feedback = output_text.split("CRITIQUE:")[-1].strip()
        
    return score, feedback

# گره اجماع و میانگین‌گیری نهایی

def consensus_node(state: ProposalState):
    print("--CONSENSUS NODE: CALCULATING AVERAGE SCORE--")
    t_score = state.get("tech_score", 0)
    b_score = state.get("biz_score", 0)
    v_score = state.get("visual_score", 0)
    
    # محاسبه میانگین قطعی نمرات

    final_score = int((t_score + b_score + v_score) / 3)
    print(f"[Consensus Result] Final Combined Score: {final_score}/100")
    
    return {"score": final_score}

In [ ]:
# ساخت گراف جدید
workflow = StateGraph(ProposalState)

# ۱. اضافه کردن تمام گره‌ها
workflow.add_node("researcher", researcher_agent)
workflow.add_node("writer", writer_agent)

# اضافه کردن سه منتقد مستقل
workflow.add_node("tech_critic", technical_critic_node)
workflow.add_node("biz_critic", business_critic_node)
workflow.add_node("visual_critic", visual_critic_node)
workflow.add_node("consensus", consensus_node)

# ۲. تعریف  اتصالات گراف
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "tech_critic")
workflow.add_edge("writer", "biz_critic")
workflow.add_edge("writer", "visual_critic")

# همگرا کردن هر سه نود منتقد
workflow.add_edge("tech_critic", "consensus")
workflow.add_edge("biz_critic", "consensus")
workflow.add_edge("visual_critic", "consensus")

# ۳. منطق شرطی برای تکرار یا پایان بر اساس خروجی نود اجماع

def should_continue(state: ProposalState):
    if state["score"] >= 95 or state.get("revision_count", 0) >= 4:
        return "end"
    return "rewrite"

workflow.add_conditional_edges(
    "consensus",
    should_continue,
    {
        "end": END,
        "rewrite": "writer"
    }
)

app = workflow.compile()

In [ ]:
# --- ۵. بخش اجرای تعاملی و پویا (Interactive Input) ---
if __name__ == "__main__":
    print("====================================================")
    print("به سیستم هوشمند و چندعامله‌ی تولید پروپوزال خوش آمدید")
    print("====================================================\n")
    
    # گرفتن موضوع یا نیازمندی به صورت لایو از کاربر
    user_topic = input("لطفاً موضوع یا نیازمندی پروژه خود را وارد کنید:\n ")
    
    if not user_topic.strip():
        print("❌ موضوع نمی‌تواند خالی باشد. برنامه متوقف شد.")
    else:
        inputs = {
            "requirements": user_topic,
            "revision_count": 0,
            "critique": []
        }
        
        print("\nفرآیند تحقیق و تولید چندبعدی آغاز شد. لطفاً صبور باشید...")
        print("----------------------------------------------------")
        
        # اجرای گراف لنگ‌گراف
        final_output = app.invoke(inputs)
        
        # ذخیره خروجی نهایی در فایل مارک‌داون
        file_name = "dynamic_proposal.md"
        with open(file_name, "w", encoding="utf-8") as file:
            file.write(final_output["current_draft"])
            
        print("\n====================================================")
        print(f"عملیات با موفقیت پایان یافت! نمره نهایی: {final_output.get('score', 0)}/100")
        print(f"پروپوزال کامل شما در فایل '{file_name}' ذخیره شد.")
        print("====================================================")